In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
from pprint import pprint

from pinecone import Pinecone

from llama_index.core import VectorStoreIndex, Settings
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

from tavily import TavilyClient

c:\Users\PIXEL-PC\OneDrive\Documents\Tatweer\work\libya-chatbot-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def tavily_search(query: str, max_results: int = 5):
    client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    return client.search(query=query, search_depth="basic", max_results=max_results)

In [3]:
def rag_retrieve(question: str, top_k: int = 5):
    retriever = index.as_retriever(similarity_top_k=top_k)
    return retriever.retrieve(question)

def verifier_supported(question: str, answer: str, retrieved_nodes, max_chunks: int = 3) -> bool:
    context = "\n\n---\n\n".join([n.node.get_text() for n in retrieved_nodes[:max_chunks]])

    prompt = f"""
Return ONLY: SUPPORTED or UNSUPPORTED.

QUESTION: {question}
ANSWER: {answer}

CONTEXT:
{context}
""".strip()

    verdict = Settings.llm.complete(prompt).text.strip().upper()
    return verdict == "SUPPORTED"

def answer_router(question: str, top_k: int = 5):
    retrieved = rag_retrieve(question, top_k=top_k)

    # if nothing retrieved -> Tavily
    if not retrieved:
        t = tavily_search(question)
        return {"mode": "tavily", "answer": t.get("answer"), "tavily": t}

    # try RAG
    resp = query_engine.query(question)
    rag_ans = str(resp)

    # verify: if not supported -> Tavily
    if not verifier_supported(question, rag_ans, retrieved):
        t = tavily_search(question)
        return {"mode": "tavily", "answer": t.get("answer"), "tavily": t}

    # supported -> return RAG with sources
    sources = []
    for sn in getattr(resp, "source_nodes", [])[:top_k]:
        md = sn.node.metadata or {}
        sources.append({
            "title": md.get("title"),
            "category": md.get("category"),
            "url": md.get("source_url"),
        })

    return {"mode": "rag", "answer": rag_ans, "sources": sources}

In [4]:
result = answer_router("من هو المحافظ الحالي لمصرف ليبيا المركزي؟")

print("MODE:", result["mode"])
print("\nANSWER:\n", result["answer"])

print("\nSOURCES:")
pprint(result.get("sources") or result.get("tavily", {}).get("results", []))

NameError: name 'index' is not defined